In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")
blindD = ak.from_parquet("Data_All_runs_BLINDED.parquet")

B_D1_IDX_FIELD            = 'Bd1Idx'
B_D2_IDX_FIELD            = 'Bd2Idx'

# Lambda daughter fields
LAMBDA_PROTON_IDX_FIELD   = 'Lambda0d1Idx'
LAMBDA_PROTON_LUND_FIELD  = 'Lambda0d1Lund'

 
PROTON_TRKIDX_FIELD       = 'pTrkIdx'
PROTON_SELECTORS_FIELD    = 'pSelectorsMap'
TRK_LUND_FIELD            = 'TRKLund'

MES_FIELD                 = 'BpostFitMes'
DELTAE_FIELD              = 'BpostFitDeltaE'

SPMODE_FIELD              = 'spmode'

plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):


    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2
def indices_to_booleans(indices, array_to_slice):
    """Convert ragged integer indices to a boolean mask over array_to_slice."""
    whole_set, in_set = ak.unzip(ak.cartesian(
        [ak.local_index(array_to_slice), indices], nested=True
    ))
    return ak.any(whole_set == in_set, axis=-1)
def ragged_element_pick(ragged_arr, flat_idx):
    """
    For each event i, return ragged_arr[i][flat_idx[i]].

    Parameters
    ----------
    ragged_arr : ak.Array, shape [n_events, var]
    flat_idx   : array-like, shape [n_events]  (integer index per event)

    Returns
    -------
    np.ndarray, shape [n_events]
    """
    flat_content = ak.flatten(ragged_arr)
    counts       = ak.num(ragged_arr)

    offsets = np.zeros(len(counts) + 1, dtype=np.int64)
    np.cumsum(ak.to_numpy(counts), out=offsets[1:])

    idx_np = ak.to_numpy(flat_idx).astype(np.int64)
    global_indices = offsets[:-1] + idx_np

    return ak.to_numpy(flat_content)[global_indices]
def build_antilambda_antimask(ak_arr, pps,
                               selector,
                               verbose=0):
    """
    Antibaryon veto for B0 -> Lambda Lambda (BNV).

    REQUIRES: single-B-candidate cut already applied (exactly one B per event).

    Returns
    -------
    charge_test : awkward array of booleans
        Per-track flag — True where a rest-of-event track is an antiproton.
    """

    index = 721

    n = len(ak_arr)

     # 1.  Squeeze B-candidate dimension
    #bd1idx = ak.to_numpy(ak_arr[B_D1_IDX_FIELD][:, 0])   # scalar per event
    bd1idx = ak.to_numpy(ak.firsts(ak_arr[B_D1_IDX_FIELD], axis=1))
    

    #bd2idx = ak.to_numpy(ak_arr[B_D2_IDX_FIELD][:, 0])
    bd2idx = ak.to_numpy(ak.firsts(ak_arr[B_D2_IDX_FIELD], axis=1))
    

    if verbose:
        print(f"bd1idx  {bd1idx[:5]}")
        print(f"bd2idx  {bd2idx[:5]}")
        print()

     
    # 2.  Chain: B daughter Lambda index -> that Lambda's proton index
     
    all_lam_proton_idx  = ak_arr[LAMBDA_PROTON_IDX_FIELD]
    all_lam_proton_lund = ak_arr[LAMBDA_PROTON_LUND_FIELD]

    # Proton-list index for each B-daughter Lambda (flat numpy, one per event)
    lam1_pidx = ragged_element_pick(all_lam_proton_idx, bd1idx)
    lam2_pidx = ragged_element_pick(all_lam_proton_idx, bd2idx)
    

    if verbose:
        print(f"lam1_pidx (proton-list index from Λ₁)  {lam1_pidx[:5]}")
        print(f"lam2_pidx (proton-list index from Λ₂)  {lam2_pidx[:5]}")
        print()

     #     qlam1 = +1 for Λ → p π⁻,  -1 for Λ̄ → p̄ π⁺
    lam1_lund = ragged_element_pick(all_lam_proton_lund, bd1idx)
    lam2_lund = ragged_element_pick(all_lam_proton_lund, bd2idx)
    

    qlam1 = np.sign(lam1_lund).astype(float)   # shapes
    qlam2 = np.sign(lam2_lund).astype(float)
    

    if verbose:
        print(f"qlam1 (charge of Λ₁ proton)  {qlam1[:5]}")
        print(f"qlam2 (charge of Λ₂ proton)  {qlam2[:5]}")
        print()

    qtrk = ak_arr[TRK_LUND_FIELD] / np.abs(ak_arr[TRK_LUND_FIELD])
    

     # 4.  Map proton-list indices to track-list indices
    trk_selector_map = ak_arr[PROTON_SELECTORS_FIELD]
    trkidx_proton    = ak_arr[PROTON_TRKIDX_FIELD]

    # Pick the track index for each signal proton (flat numpy)
    lam1_trkidx_flat = ragged_element_pick(trkidx_proton, lam1_pidx)
    lam2_trkidx_flat = ragged_element_pick(trkidx_proton, lam2_pidx)

    # Wrap as length-1 ragged for indices_to_booleans
    lam1_trkidx = ak.unflatten(lam1_trkidx_flat, 1)
    lam2_trkidx = ak.unflatten(lam2_trkidx_flat, 1)
    if verbose:
        print(f"lam1_trkidx  {lam1_trkidx[:5]}")
        print(f"lam2_trkidx  {lam2_trkidx[:5]}")
        print()

    bool_idx1 = indices_to_booleans(lam1_trkidx, trk_selector_map)
    bool_idx2 = indices_to_booleans(lam2_trkidx, trk_selector_map)


    if verbose:
        print(f"bool_idx1 (Λ₁ proton)  {bool_idx1[:3]}")
        print(f"bool_idx2 (Λ₂ proton)  {bool_idx2[:3]}")
        print()

    
    # 5.  PID selection on ALL proton candidates  
    pbits     = calculate_bits_for_PID_selector(None, trk_selector_map)
    mask_bool = mask_PID_selection(pbits, selector, pps)

    if verbose:
        print(f"Tracks passing {selector}:  {ak.sum(ak.num(pbits[mask_bool]))}")
        print()

     # 6.  Charge test on REST-OF-EVENT proton candidates
     
    other_mask  = ~(bool_idx1 | bool_idx2) & mask_bool
    charge_test = qtrk[other_mask] == -qlam1


    if verbose:
        print(f"charge_test  {charge_test[:3]}")

   
    have_opp       = ak.num(charge_test[charge_test])
    nhave_opp      = len(have_opp[have_opp > 0])
    ndont_have_opp = len(have_opp[have_opp == 0])

    return charge_test
def apply_antibaryon_veto(ak_arr, pps, selector = 'LooseKMProtonSelection'):
    """Return a per-event boolean: True = passes veto (no antiproton found)."""
    ct       = build_antilambda_antimask(ak_arr, pps, selector=selector, verbose=0)
    have_opp = ak.num(ct[ct])
    return have_opp == 0

pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")

pidChoiceDict = {}
for cutI in [1]:
    for pidChoice in [1]:# 157, 202]:#,1 112, 256, 121]:
        if pidChoice == 1:
            myPMask = "SuperLooseKMProtonSelection"
            myPShort = "SL Pro"
            myPiMask = "SuperLooseKMPionMicroSelection"
            myPiShort = 'SL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 157:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 202:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "TightKMPionMicroSelection"
            myPiShort = 'T   Pi'
            pidChoiceDict[pidChoice] =(myPShort,myPiShort)
        elif pidChoice == 112:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "LooseKMPionMicroSelection"
            myPiShort = 'L   Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 256:
            myPMask = "LooseKMProtonSelection"
            myPShort = "L   Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 121:
            myPMask = "SuperLooseKMProtonSelection"
            myPShort = "SL Pro"
            myPiMask = "SuperTightKMPionMicroSelection"
            myPiShort = 'ST Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -1:
            myPMask = "TightKMProtonSelection"
            myPShort = "T   Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -2:
            myPMask = "VeryTightKMProtonSelection"
            myPShort = "VT Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -3:
            myPMask = "SuperTightKMProtonSelection"
            myPShort = "ST Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 0:
            myPMask =  "SuperLooseKMProtonSelection"
            myPiMask = "SuperLooseKMPionMicroSelection"
            pidChoiceDict[pidChoice] = ("None","None")

        #numBCan = data['nB']
        #numBCanBlind = blindD['nB']

        #bCanMask = (numBCan==1)
        bMassEmptyMask = (ak.num(data['BMass'])>0)
        #bCanMaskBlind = (numBCanBlind==1)
        bMassEmptyMaskBlind = (ak.num(blindD['BMass'])>0)

        d1b = data[bMassEmptyMask]
        d1Blind = blindD[bMassEmptyMaskBlind]



        indices_and_maps = get_info_for_PID_masks(d1b, verbosity=0)
        indices_and_mapsBlind = get_info_for_PID_masks(d1Blind, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                    lam1p_selector=myPMask, \
                    lam1pi_selector=myPiMask, \
                    lam2p_selector=myPMask, \
                    lam2pi_selector=myPiMask, \
                    verbosity=0)
        
        mask_bool_protonL1Blind, mask_bool_pionL1Blind,mask_bool_protonL2Blind, mask_bool_pionL2Blind = PID_masks(indices_and_mapsBlind, \
                    lam1p_selector=myPMask, \
                    lam1pi_selector=myPiMask, \
                    lam2p_selector=myPMask, \
                    lam2pi_selector=myPiMask, \
                    verbosity=0)
        


        
        maxSigCut,minSigCut = (10,10)
        

        mes = d1b['BpostFitMes']
        de = d1b['BpostFitDeltaE']


        mesBlind = d1Blind['BpostFitMes']
        deBlind = d1Blind['BpostFitDeltaE']

        meslo = 5.2
        mesloS = 5.27
        meshiS = 5.3
        DeltaElo = -0.2
        DeltaEhi = 0.2
        DeltaEloS = -0.07
        DeltaEhiS = 0.07

        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2
        mask_pidBlind = mask_bool_protonL1Blind & mask_bool_pionL1Blind & mask_bool_protonL2Blind & mask_bool_pionL2Blind

        preSel_fit = (mes>=meslo)&(de<=DeltaEhi)&(de>=DeltaElo)
        preSel_fitBlind = (mesBlind>meslo)&(deBlind<DeltaEhi)&(deBlind>DeltaElo)

        mask_sig = (mes>=mesloS) & (mes<=meshiS) & (de>=DeltaEloS) & (de<=DeltaEhiS)

        mask_sideband = (((mes<mesloS) | (mes>meshiS)) | ((de<DeltaEloS) | (de>DeltaEhiS)))
        mask_sidebandBlind = (((mesBlind<mesloS) | (mesBlind>meshiS)) | ((deBlind<DeltaEloS) | (deBlind>DeltaEhiS)))

        lamconflsig = d1b['Lambda0postFitFlightSignificance']
        lamuncmass = d1b['Lambda0_unc_Mass']
        BCTT = d1b['BCosThetaT']
        BCTS = d1b['BCosThetaS']

        lamconflsigBlind = d1Blind['Lambda0postFitFlightSignificance']
        lamuncmassBlind = d1Blind['Lambda0_unc_Mass']
        BCTTBlind = d1Blind['BCosThetaT']
        BCTSBlind = d1Blind['BCosThetaS']

        #TMA = d1b['thrustMagAll']
        #BP2 = d1b['BLegendreP2']


        lammass_world_average = 1.115683
        width = 0.003
        lo = lammass_world_average - width #1.112683
        hi = lammass_world_average + width #1.118683
            

        mask_fl = (lamconflsig>=minSigCut)
        mask_upper_fl = (lamconflsig>=maxSigCut)
        flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))


        """
        flSigMax = ak.max(lamconflsig, axis=1)
        flSigMin = ak.min(lamconflsig, axis=1)

        print(flSigMax[:10])
        print(flSigMin[:10])

        flightSigMask = (flSigMax>=10)&(flSigMin>=10)
        """

        mask_flBlind = (lamconflsigBlind>=minSigCut)
        mask_upper_flBlind = (lamconflsigBlind>=maxSigCut)
        flightSigMaskBlind = (ak.all(mask_flBlind,axis=1))&(ak.any(mask_upper_flBlind, axis=1))
        

        lamuncmassMask = (lamuncmass[:,0]>=lo)&(lamuncmass[:,0]<=hi)&(lamuncmass[:,1]>=lo)&(lamuncmass[:,1]<=hi)
        
        lamuncmassMaskBlind = (lamuncmassBlind[:,0]>=lo)&(lamuncmassBlind[:,0]<=hi)&(lamuncmassBlind[:,1]>=lo)&(lamuncmassBlind[:,1]<=hi)
        

        vetoMask = apply_antibaryon_veto(d1b, pps)
        
        
        vetoMaskBlind = apply_antibaryon_veto(d1Blind, pps)

        bcts_cut = 95
        bctt_cut = 82
        bcttMask = (BCTT<bctt_cut/100) 
        bctsMask = (BCTS<bcts_cut/100)



        finalMask = preSel_fit & flightSigMask & vetoMask & mask_pid & lamuncmassMask & bcttMask & bctsMask

        bcttMaskBlind = (BCTTBlind<bctt_cut/100) 
        bctsMaskBlind = (BCTSBlind<bcts_cut/100)

        finalMaskBlind = preSel_fitBlind & flightSigMaskBlind & mask_pidBlind & lamuncmassMaskBlind & vetoMaskBlind & bcttMaskBlind & bctsMaskBlind

        in_signal = preSel_fit&mask_sig
        in_sideband = preSel_fit&mask_sideband


        in_sidebandBlind = preSel_fitBlind&mask_sidebandBlind

        scaleD = {"998": 424.3/1719, "1005":424.3/868.3, "-999":1,'1235':424.3/1289,'1237':424.3/1306}
        for nTRK_cut in range(0,21):
            effDict = {}
            n_srD = {}
            n_sbD = {}
            trkMask = (d1b['nTRK']>nTRK_cut)
            for sp in ['998','1005','-999','1235','1237']:
                spmask = (d1b['spmode']==sp)
                cut_mask = finalMask&spmask&trkMask

                effDict[sp]=ak.sum(cut_mask)/(ak.sum(data['spmode']==sp))
                n_srD[sp]=ak.sum(cut_mask&in_signal)*scaleD[sp]
                n_sbD[sp]=ak.sum(cut_mask&in_sideband)*scaleD[sp]
            
            trkMask = (d1Blind['nTRK']>nTRK_cut/10)
            cut_maskBlind = finalMaskBlind&trkMask

            bkgBlind = ak.sum(cut_maskBlind&in_sidebandBlind)


            #n_srD['998']+n_srD['1005']+n_srD['1235']+n_srD["1237"]
            #n_srD['998']+n_srD['1005']+n_srD['1235']+n_srD["1237"]+n_sbD['998']+n_sbD['1005']+n_sbD['1235']+n_sbD["1237"]
            #bkgBlind


        
            print(f"nTRK>{nTRK_cut:2.0f} / Punzi (SR bkg): {effDict["-999"]/(np.sqrt(n_srD['998']+n_srD['1005']+n_srD['1235']+n_srD["1237"])+2):.4f} / Punzi (SR+SB bkg): {effDict["-999"]/(np.sqrt(n_srD['998']+n_srD['1005']+n_srD['1235']+n_srD["1237"]+n_sbD['998']+n_sbD['1005']+n_sbD['1235']+n_sbD["1237"])+2):.4f} / Punzi (blindData): {effDict["-999"]/(np.sqrt(bkgBlind)+2):.4f} / sigEff: {effDict["-999"]:.4f} / 998Eff:  {effDict['998']:.4f} / 1005Eff: {effDict['1005']:.4f} / b_sr: {n_srD['998']+n_srD['1005']+n_srD['1235']+n_srD["1237"]:.2f} / b_sb: {n_sbD['998']+n_sbD['1005']+n_sbD['1235']+n_sbD["1237"]:.2f}")
